# MIMIC-IV Sepsis Prediction Machine Learning Pipeline
This notebook reproduces and extends the methodology of the Sepsis Prediction SOTA paper (*Evaluating deep learning sepsis prediction models in ICUs under distribution shift: a multi-centre retrospective cohort study*, Tranchellini et al., 2026).

It connects to Google BigQuery and loads both:
1. **Sequential hourly features** (6 rows per stay) for deep learning models (**PyTorch CNN and LSTM**).
2. **Aggregated tabular features** (1 row per stay) for traditional ML models (**XGBoost, LightGBM, and Random Forest**).

It evaluates both sets of models at prediction horizons of **6, 12, 18, and 24 hours before sepsis onset**, providing classification metrics (including Normalized AUPRC) and **SHAP explainability**.

In [ ]:
# Install required packages
!pip install lightgbm xgboost shap pandas-gbq scikit-learn matplotlib seaborn torch

In [ ]:
# ---------------------------------------------------------------------------
# STEP 1: GCP AUTHENTICATION
# ---------------------------------------------------------------------------
from google.colab import auth
auth.authenticate_user()
print("Authenticated successfully!")

In [ ]:
# ---------------------------------------------------------------------------
# STEP 2: LIBRARIES AND CONFIGURATION
# ---------------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
import shap
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ID = 'mimic-research-490610'
WINDOWS = ['6h', '12h', '18h', '24h']
TARGET_COLUMN = 'label'
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

## Step 3: Traditional Machine Learning Models (Tabular Pipeline)
We train and evaluate XGBoost, LightGBM, and Random Forest on the aggregated tabular feature tables (1 row per stay).

In [ ]:
all_window_results_ml = {}
trained_models_ml = {}
imputers_ml = {}
scalers_ml = {}
feature_columns_dict_ml = {}

for window in WINDOWS:
    print(f"\n=====================================================================")
    print(f" TRAINING ML MODELS FOR HORIZON: {window} ")
    print(f"=====================================================================")
    
    # 1. Load data from BigQuery
    print(f"--> Loading final_sepsis_features_{window} from BigQuery...")
    query = f"SELECT * FROM `mimic-research-490610.sepsis_cohort.final_sepsis_features_{window}`"
    df = pd.read_gbq(query, project_id=PROJECT_ID)
    print(f"Loaded dataset shape: {df.shape}")
    
    # 2. Preprocessing
    df['gender'] = df['gender'].map({'M': 1.0, 'F': 0.0}).fillna(0.5)
    
    drop_cols = ['patientunitstayid', TARGET_COLUMN]
    feature_cols = [col for col in df.columns if col not in drop_cols]
    
    X = df[feature_cols].copy()
    y = df[TARGET_COLUMN].copy()
    
    # Drop columns that are 100% missing
    X = X.dropna(how='all', axis=1)
    feature_cols = list(X.columns)
    feature_columns_dict_ml[window] = feature_cols
    
    # 3. Train-test split (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=SEED
    )
    
    # 4. Imputation (MICE)
    print("--> Imputing missing values using MICE (IterativeImputer)...")
    imputer = IterativeImputer(random_state=SEED, max_iter=10)
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    imputers_ml[window] = imputer
    
    # 5. Scaling
    print("--> Standardizing continuous features...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    scalers_ml[window] = scaler
    
    X_train_df = pd.DataFrame(X_train_scaled, columns=feature_cols)
    X_test_df = pd.DataFrame(X_test_scaled, columns=feature_cols)
    
    # 6. Calculate class weight for imbalance
    neg_count = np.sum(y_train == 0)
    pos_count = np.sum(y_train == 1)
    scale_weight = neg_count / pos_count if pos_count > 0 else 1.0
    print(f"Target distribution - Control (0): {np.sum(y==0)}, Sepsis (1): {np.sum(y==1)}")
    
    # 7. Initialize and train models
    models = {
        "XGBoost": XGBClassifier(scale_pos_weight=scale_weight, random_state=SEED, eval_metric='logloss'),
        "LightGBM": lgb.LGBMClassifier(scale_pos_weight=scale_weight, random_state=SEED),
        "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=SEED)
    }
    
    window_results = {}
    trained_models_ml[window] = {}
    
    plt.figure(figsize=(8, 6))
    for name, model in models.items():
        print(f"Training {name}...")
        model.fit(X_train_df, y_train)
        trained_models_ml[window][name] = model
        
        y_pred = model.predict(X_test_df)
        y_prob = model.predict_proba(X_test_df)[:, 1]
        
        # Metrics
        auc = metrics.roc_auc_score(y_test, y_prob)
        acc = metrics.accuracy_score(y_test, y_pred)
        ap = metrics.average_precision_score(y_test, y_prob)
        f1_w = metrics.f1_score(y_test, y_pred, average='weighted')
        sens = metrics.recall_score(y_test, y_pred)
        spec = metrics.recall_score(y_test, y_pred, pos_label=0)
        bal_acc = metrics.balanced_accuracy_score(y_test, y_pred)
        
        # Calculate Normalized AUPRC
        baseline = np.sum(y_test == 1) / len(y_test)
        norm_auprc = ap / baseline if baseline > 0 else 1.0
        
        window_results[name] = {
            "ROC-AUC": auc,
            "AUPRC": ap,
            "Normalized AUPRC": norm_auprc,
            "Accuracy": acc,
            "Weighted F1": f1_w,
            "Sensitivity": sens,
            "Specificity": spec,
            "Balanced Accuracy": bal_acc
        }
        
        fpr, tpr, _ = metrics.roc_curve(y_test, y_prob)
        plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
        
    plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves - Tabular Sepsis Prediction ({window} window)")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()
    
    df_window = pd.DataFrame(window_results).T
    print(f"\nResults for {window} window:")
    display(df_window)
    
    all_window_results_ml[window] = window_results

## Step 4: Deep Learning Models (PyTorch Sequential Pipeline)
We load sequential hourly data (6 rows per stay), split data at the stay level to prevent leakage, perform sequential imputation, and train fully operational **PyTorch CNN** and **LSTM** classifiers.

In [ ]:
class PyTorchCNN(nn.Module):
    def __init__(self, num_features, window_size=6, dropout=0.5):
        super(PyTorchCNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=256, kernel_size=1, padding=1)
        self.bn1 = nn.BatchNorm1d(256)
        self.conv2 = nn.Conv1d(in_channels=256, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=16, kernel_size=5, padding=1)
        self.bn3 = nn.BatchNorm1d(16)
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.dropout = nn.Dropout(dropout)
        
        # window_size is 6, after 3 Conv1d (with padding=1) and 1 MaxPool(kernel_size=2)
        # we calculate output length of temporal dimension dynamically
        self.fc1 = nn.Linear(16 * 3, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x, return_features=False):
        # Input x shape: (batch_size, num_features, window_size)
        x = F.leaky_relu(self.bn1(self.conv1(x)))
        x = F.leaky_relu(self.bn2(self.conv2(x)))
        x = F.leaky_relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        features = x.view(x.size(0), -1)  # Flatten
        x = self.dropout(features)
        x = F.leaky_relu(self.fc1(x))
        x = self.fc2(x)
        if return_features:
            return x.squeeze(-1), features
        return x.squeeze(-1)

class PyTorchLSTM(nn.Module):
    def __init__(self, input_size, lstm_units=[64, 32], dropout_rate=0.5):
        super(PyTorchLSTM, self).__init__()
        self.lstm1 = nn.LSTM(input_size, lstm_units[0], batch_first=True)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.bn1 = nn.BatchNorm1d(lstm_units[0])
        
        self.lstm2 = nn.LSTM(lstm_units[0], lstm_units[1], batch_first=True)
        self.dropout2 = nn.Dropout(dropout_rate)
        self.bn2 = nn.BatchNorm1d(lstm_units[1])
        
        self.dense1 = nn.Linear(lstm_units[1], 16)
        self.dense2 = nn.Linear(16, 1)

    def forward(self, x, return_features=False):
        # Input x shape: (batch_size, window_size, num_features)
        x, _ = self.lstm1(x)
        x = self.dropout1(x)
        x = self.bn1(x.transpose(1, 2)).transpose(1, 2)
        
        x, _ = self.lstm2(x)
        x = self.dropout2(x)
        x = self.bn2(x.transpose(1, 2)).transpose(1, 2)
        
        features = x[:, -1, :]
        x = F.leaky_relu(self.dense1(features))
        x = self.dense2(x)
        if return_features:
            return x.squeeze(-1), features
        return x.squeeze(-1)

In [ ]:
def train_pytorch_model(model, train_loader, val_loader, pos_weight, num_epochs=30, patience=5, device='cpu'):
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.9, patience=3)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], dtype=torch.float32).to(device))
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = pickle.loads(pickle.dumps(model.state_dict())) # Deep copy weights
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
                
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

In [ ]:
def evaluate_pytorch_model(model, test_loader, device='cpu'):
    model.eval()
    y_true = []
    y_probs = []
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            outputs = model(batch_x)
            probs = torch.sigmoid(outputs)
            y_true.extend(batch_y.numpy())
            y_probs.extend(probs.cpu().numpy())
            
    y_true = np.array(y_true)
    y_probs = np.array(y_probs)
    y_pred = (y_probs >= 0.5).astype(float)
    
    auc = metrics.roc_auc_score(y_true, y_probs)
    ap = metrics.average_precision_score(y_true, y_probs)
    acc = metrics.accuracy_score(y_true, y_pred)
    f1_w = metrics.f1_score(y_true, y_pred, average='weighted')
    sens = metrics.recall_score(y_true, y_pred)
    spec = metrics.recall_score(y_true, y_pred, pos_label=0)
    bal_acc = metrics.balanced_accuracy_score(y_true, y_pred)
    
    baseline = np.sum(y_true == 1) / len(y_true)
    norm_auprc = ap / baseline if baseline > 0 else 1.0
    
    return {
        "ROC-AUC": auc,
        "AUPRC": ap,
        "Normalized AUPRC": norm_auprc,
        "Accuracy": acc,
        "Weighted F1": f1_w,
        "Sensitivity": sens,
        "Specificity": spec,
        "Balanced Accuracy": bal_acc
    }, y_probs

In [ ]:
all_window_results_dl = {}
trained_models_dl = {}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for window in WINDOWS:
    print(f"\n=====================================================================")
    print(f" TRAINING DEEP LEARNING MODELS FOR HORIZON: {window} ")
    print(f"=====================================================================")
    
    # 1. Load sequential data from BigQuery
    query = f"SELECT * FROM `mimic-research-490610.sepsis_cohort.final_sepsis_seq_features_{window}` ORDER BY patientunitstayid, hour_idx"
    df_seq = pd.read_gbq(query, project_id=PROJECT_ID)
    print(f"Loaded sequential dataset shape: {df_seq.shape}")
    
    # 2. Preprocessing
    df_seq['gender'] = df_seq['gender'].map({'M': 1.0, 'F': 0.0}).fillna(0.5)
    
    feature_cols = [col for col in df_seq.columns if col not in ['patientunitstayid', 'hour_idx', TARGET_COLUMN]]
    
    # Exclude entirely null features
    df_seq_clean = df_seq.dropna(how='all', axis=1)
    feature_cols = [col for col in df_seq_clean.columns if col not in ['patientunitstayid', 'hour_idx', TARGET_COLUMN]]
    
    # Get stay-level labels for stratified split
    stay_labels = df_seq.groupby('patientunitstayid')[TARGET_COLUMN].first()
    stay_ids = stay_labels.index.values
    labels = stay_labels.values
    
    # Split stay IDs to prevent data leakage
    train_stays, test_stays, y_train_stays, y_test_stays = train_test_split(
        stay_ids, labels, test_size=0.20, stratify=labels, random_state=SEED
    )
    train_stays, val_stays, _, _ = train_test_split(
        train_stays, y_train_stays, test_size=0.10, stratify=y_train_stays, random_state=SEED
    )
    
    # Filter dataframes
    df_train = df_seq[df_seq['patientunitstayid'].isin(train_stays)].copy()
    df_val = df_seq[df_seq['patientunitstayid'].isin(val_stays)].copy()
    df_test = df_seq[df_seq['patientunitstayid'].isin(test_stays)].copy()
    
    # Imputation: Forward fill within each stay, then fill remaining with train mean
    print("--> Performing forward-fill and training-mean imputation...")
    df_train[feature_cols] = df_train.groupby('patientunitstayid')[feature_cols].ffill()
    df_val[feature_cols] = df_val.groupby('patientunitstayid')[feature_cols].ffill()
    df_test[feature_cols] = df_test.groupby('patientunitstayid')[feature_cols].ffill()
    
    train_means = df_train[feature_cols].mean()
    df_train[feature_cols] = df_train[feature_cols].fillna(train_means)
    df_val[feature_cols] = df_val[feature_cols].fillna(train_means)
    df_test[feature_cols] = df_test[feature_cols].fillna(train_means)
    
    # Scaling
    print("--> Standardizing sequential features...")
    scaler = StandardScaler()
    df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])
    df_val[feature_cols] = scaler.transform(df_val[feature_cols])
    df_test[feature_cols] = scaler.transform(df_test[feature_cols])
    
    # Reshape into 3D tensors: shape (N, 6, num_features)
    def to_3d_tensor(df, stays):
        N = len(stays)
        F_dim = len(feature_cols)
        x_3d = np.zeros((N, 6, F_dim))
        y_arr = np.zeros(N)
        for idx, stay_id in enumerate(stays):
            stay_data = df[df['patientunitstayid'] == stay_id].sort_values('hour_idx')
            # If there are fewer than 6 hours recorded, we pad with zeros
            h_len = min(6, len(stay_data))
            x_3d[idx, :h_len, :] = stay_data[feature_cols].values[:h_len]
            y_arr[idx] = stay_data[TARGET_COLUMN].values[0]
        return torch.tensor(x_3d, dtype=torch.float32), torch.tensor(y_arr, dtype=torch.float32)
        
    X_train_3d, y_train_3d = to_3d_tensor(df_train, train_stays)
    X_val_3d, y_val_3d = to_3d_tensor(df_val, val_stays)
    X_test_3d, y_test_3d = to_3d_tensor(df_test, test_stays)
    
    # Create loaders
    train_loader = DataLoader(TensorDataset(X_train_3d, y_train_3d), batch_size=32, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_3d, y_val_3d), batch_size=32, shuffle=False)
    test_loader = DataLoader(TensorDataset(X_test_3d, y_test_3d), batch_size=32, shuffle=False)
    
    # Calculate class pos_weight
    pos_weight = (y_train_3d == 0).sum().item() / (y_train_3d == 1).sum().item()
    num_features = len(feature_cols)
    
    # 3. Train PyTorch LSTM
    print("Training PyTorch LSTM...")
    lstm_model = PyTorchLSTM(input_size=num_features).to(device)
    lstm_model = train_pytorch_model(lstm_model, train_loader, val_loader, pos_weight, device=device)
    lstm_metrics, _ = evaluate_pytorch_model(lstm_model, test_loader, device=device)
    
    # 4. Train PyTorch CNN (expects shape (batch, num_features, window_size))
    # We transpose input from (batch, 6, features) to (batch, features, 6)
    class transposed_cnn(nn.Module):
        def __init__(self, cnn_core):
            super().__init__()
            self.core = cnn_core
        def forward(self, x, return_features=False):
            # transpose (batch, 6, features) to (batch, features, 6)
            x = x.transpose(1, 2)
            return self.core(x, return_features)
            
    print("Training PyTorch CNN...")
    cnn_core = PyTorchCNN(num_features=num_features).to(device)
    cnn_model = transposed_cnn(cnn_core)
    cnn_model = train_pytorch_model(cnn_model, train_loader, val_loader, pos_weight, device=device)
    cnn_metrics, _ = evaluate_pytorch_model(cnn_model, test_loader, device=device)
    
    window_results = {
        "LSTM": lstm_metrics,
        "CNN": cnn_metrics
    }
    
    trained_models_dl[window] = {
        "LSTM": lstm_model,
        "CNN": cnn_model
    }
    
    df_window_dl = pd.DataFrame(window_results).T
    print(f"\nDeep Learning Results for {window} window:")
    display(df_window_dl)
    
    all_window_results_dl[window] = window_results

## Step 5: Consolidated Sepsis Model Performance Comparison
We compare traditional ML and deep learning model results side-by-side across all horizons.

In [ ]:
for metric in ["ROC-AUC", "Normalized AUPRC", "Sensitivity", "Balanced Accuracy"]:
    print(f"\n=====================================================================")
    print(f" COMPARISON FOR METRIC: {metric} ")
    print(f"=====================================================================")
    
    comparison_data = {}
    for window in WINDOWS:
        window_dict = {}
        for model in ["XGBoost", "LightGBM", "Random Forest"]:
            window_dict[model] = all_window_results_ml[window][model][metric]
        for model in ["LSTM", "CNN"]:
            window_dict[model] = all_window_results_dl[window][model][metric]
        comparison_data[window] = window_dict
        
    df_comp = pd.DataFrame(comparison_data)
    display(df_comp)

## Step 6: SHAP Explainability (Aggregated and Sequential)
We generate feature contributions using SHAP for the XGBoost model at each horizon window.

In [ ]:
for window in WINDOWS:
    print(f"\nGenerating SHAP summary plot for the {window} XGBoost model...")
    xgb_model = trained_models_ml[window]["XGBoost"]
    feature_cols = feature_columns_dict_ml[window]
    
    # Load a small batch for SHAP analysis
    df_w = pd.read_gbq(f"SELECT * FROM `mimic-research-490610.sepsis_cohort.final_sepsis_features_{window}` LIMIT 500", project_id=PROJECT_ID)
    df_w['gender'] = df_w['gender'].map({'M': 1.0, 'F': 0.0}).fillna(0.5)
    X_w = df_w[feature_cols].copy()
    
    X_w_imp = imputers_ml[window].transform(X_w)
    X_w_scaled = scalers_ml[window].transform(X_w_imp)
    X_w_df = pd.DataFrame(X_w_scaled, columns=feature_cols)
    
    explainer = shap.TreeExplainer(xgb_model, feature_perturbation='tree_path_dependent')
    shap_values = explainer.shap_values(X_w_df)
    
    if isinstance(shap_values, list) and len(shap_values) == 2:
        shap_values_to_plot = shap_values[1]
    else:
        shap_values_to_plot = shap_values
        
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_to_plot, X_w_df, show=False)
    plt.title(f"SHAP Feature Importance Summary - Horizon {window}", fontsize=14)
    plt.tight_layout()
    plt.show()